# BiLSTM (+Attention) for Protein Secondary Structure Prediction (Q3/Q8)

Self-contained Colab notebook: loads data, preprocesses, trains a BiLSTM with optional self-attention refinement, and evaluates with Q3/Q8 accuracy, per-class PRF, confusion matrix, and SOV. Optimized for A100 with AMP.

In [8]:
# Optional: install dependencies
# If running in Colab and you don't have deps, uncomment:
# !pip -q install torch numpy pandas scikit-learn pyyaml tqdm matplotlib
import os, sys, math, random, gc, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
torch.backends.cudnn.benchmark = True

PyTorch: 2.9.0+cu128 | CUDA: True | GPU: NVIDIA A100-SXM4-40GB


## Config

In [16]:
CFG = {
    'seed': 42,
    'data': {
        'sequences_csv': '/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-pdb-intersect-pisces.csv',
        'labels_csv': '/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-ss.cleaned.csv',
        'id_column': 'pdb_id',
        'seq_column': 'seq',
        'label_column_q3': 'sst3',
        'label_column_q8': 'sst8',
        'target': 'q8',  # 'q3' or 'q8'
        'max_len': 700,
        'max_seq_len_filter': 800,
        'train_fraction': 0.8,
        'val_fraction': 0.1,
        'length_stratified_split': False,
    },
    'model': {
        'vocab': 'ACDEFGHIKLMNPQRSTVWYX',  # X unknown; PAD=0
        'embedding_dim': 128,
        'hidden_dim': 256,
        'dropout': 0.3,
        'layers': 2,
        'attention': True,            # refinement: self-attention block
        'attn_heads': 4,             # multihead attention heads
        'attn_dropout': 0.1,
        'label_smoothing': 0.05,     # refinement: small label smoothing
    },
    'training': {
        'epochs': 40,
        'batch_size': 32,
        'lr': 1e-3,
        'optimizer': 'adamw',
        'weight_decay': 1e-2,
        'grad_clip_norm': 1.0,
        'mixed_precision': True,
        'num_workers': 2,
        'pin_memory': True,
        'early_stopping_patience': 5,
        'reduce_lr_on_plateau': {
            'enabled': True, 'factor': 0.5, 'patience': 3
        }
    },
    'output': {
        'checkpoints_dir': 'checkpoints',
        'best_model_path': 'checkpoints/bilstm_${target}.pt'
    }
}
PAD_IDX = 0
IGNORE_INDEX = -100
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(CFG['output']['checkpoints_dir'], exist_ok=True)
print('Target:', CFG['data']['target'], '| Device:', device)

Target: q8 | Device: cuda


## Utilities

In [17]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

def build_vocab(vocab_str):
    return {aa: i+1 for i, aa in enumerate(vocab_str)}  # 0 reserved for PAD

def encode_sequence(seq, vocab, unk_token='X'):
    seq = (seq or '').strip().upper()
    return [vocab.get(ch, vocab.get(unk_token, len(vocab))) for ch in seq]

Q3_MAP = {'H':0,'E':1,'C':2}
Q8_MAP = {'H':0,'G':1,'I':2,'E':3,'B':4,'T':5,'S':6,'C':7}

def encode_labels(lbls, scheme='q8'):
    lbls = (lbls or '').strip().upper()
    m = Q8_MAP if scheme=='q8' else Q3_MAP
    return [m[c] for c in lbls if c in m]

def pad_batch(seqs, labels, max_len):
    batch_max = min(max_len, max(len(s) for s in seqs))
    bs = len(seqs)
    x = torch.full((bs, batch_max), PAD_IDX, dtype=torch.long)
    y = torch.full((bs, batch_max), IGNORE_INDEX, dtype=torch.long)
    mask = torch.zeros((bs, batch_max), dtype=torch.bool)
    lengths = []
    for i,(s,l) in enumerate(zip(seqs,labels)):
        n = min(len(s), batch_max)
        x[i,:n] = torch.tensor(s[:n], dtype=torch.long)
        y[i,:n] = torch.tensor(l[:n], dtype=torch.long)
        mask[i,:n] = True
        lengths.append(n)
    return x,y,mask,torch.tensor(lengths,dtype=torch.long)

def masked_accuracy(logits, labels, mask):
    with torch.no_grad():
        preds = logits.argmax(dim=-1)
        valid = mask & (labels >= 0)
        correct = (preds[valid] == labels[valid]).sum().item()
        total = valid.sum().item()
        return correct/total if total>0 else 0.0

def _segments(labels, cls):
    segs = []
    start = None
    for i,v in enumerate(labels):
        if v==cls and start is None: start=i
        elif v!=cls and start is not None:
            segs.append((start, i-1)); start=None
    if start is not None: segs.append((start, len(labels)-1))
    return segs

def _overlap(a,b):
    s = max(a[0], b[0]); e = min(a[1], b[1])
    return max(0, e-s+1)

def _length(seg):
    return seg[1]-seg[0]+1

def sov_score(y_true, y_pred, num_classes):
    total_len = 0; accum = 0.0
    for cls in range(num_classes):
        true_segs = _segments(y_true, cls)
        pred_segs = _segments(y_pred, cls)
        for t in true_segs:
            t_len = _length(t); total_len += t_len
            overlaps = [p for p in pred_segs if _overlap(t,p)>0]
            if not overlaps: continue
            max_sov = 0.0
            for p in overlaps:
                minov = _overlap(t,p); maxov = max(_length(t), _length(p))
                if minov==0: continue
                delta = min(maxov-minov, minov, _length(t)//2, _length(p)//2)
                sov = (minov+delta)/maxov * t_len
                max_sov = max(max_sov, sov)
            accum += max_sov
    return accum/total_len if total_len>0 else 0.0

def evaluate_logits(all_logits, all_labels, all_masks, num_classes):
    preds = all_logits.argmax(dim=-1)
    valid = all_masks & (all_labels >= 0)
    y_true = all_labels[valid].cpu().numpy()
    y_pred = preds[valid].cpu().numpy()
    acc = (y_true==y_pred).mean() if y_true.size>0 else 0.0
    p,r,f1,supp = precision_recall_fscore_support(y_true, y_pred, labels=list(range(num_classes)), zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    sov = sov_score(y_true, y_pred, num_classes)
    return {'accuracy': float(acc), 'precision': p, 'recall': r, 'f1': f1, 'support': supp, 'confusion_matrix': cm, 'sov': float(sov)}

set_seed(CFG['seed'])

In [18]:
seq_df = pd.read_csv(CFG['data']['sequences_csv'])
lab_df = pd.read_csv(CFG['data']['labels_csv'])

In [19]:
seq_df.head()

,pdb_id,chain_code,seq,sst8,sst3,len,has_nonstd_aa,Exptl.,resolution,R-factor,FreeRvalue
0,1FV1,F,NPVVHFFKNIVTPRTPPPSQ,CCCCCBCCCCCCCCCCCCCC,CCCCCECCCCCCCCCCCCCC,20,False,XRAY,1.90,0.23,0.27
1,1LM8,H,DLDLEMLAPYIPMDDDFQLR,CCCCCCCCCBCCSCCCEECC,CCCCCCCCCECCCCCCEECC,20,False,XRAY,1.85,0.20,0.24
2,1O06,A,EEDPDLKAAIQESLREAEEA,CCCHHHHHHHHHHHHHHHTC,CCCHHHHHHHHHHHHHHHCC,20,False,XRAY,1.45,0.19,0.22
3,1QOW,D,CTFTLPGGGGVCTLTSECI*,CCTTSCTTCSSTTSSTTCCC,CCCCCCCCCCCCCCCCCCCC,20,True,XRAY,1.06,0.14,1.00
4,1RDQ,I,TTYADFIASGRTGRRNAIHD,CHHHHHHTSSCSSCCCCEEC,CHHHHHHCCCCCCCCCCEEC,20,False,XRAY,1.26,0.13,0.16


In [20]:
lab_df.head()

,pdb_id,chain_code,seq,sst8,sst3,len,has_nonstd_aa
0,1A30,C,EDL,CBC,CEC,3,False
1,1B05,B,KCK,CBC,CEC,3,False
2,1B0H,B,KAK,CBC,CEC,3,False
3,1B1H,B,KFK,CBC,CEC,3,False
4,1B2H,B,KAK,CBC,CEC,3,False


## Load and preprocess data

In [21]:
# Load CSVs and merge
seq_df = pd.read_csv(CFG['data']['sequences_csv'], usecols=[CFG['data']['id_column'], CFG['data']['seq_column']])
lab_df = pd.read_csv(CFG['data']['labels_csv'], usecols=[CFG['data']['id_column'], CFG['data']['label_column_q3'], CFG['data']['label_column_q8']])
df = pd.merge(seq_df, lab_df, on=CFG['data']['id_column'], how='inner')
print('Merged:', df.shape)
# Clean
seq_col = CFG['data']['seq_column']
lbl_col = CFG['data']['label_column_q8'] if CFG['data']['target']=='q8' else CFG['data']['label_column_q3']
df = df.dropna(subset=[seq_col, lbl_col]).copy()
df[seq_col] = df[seq_col].str.upper()
df[lbl_col] = df[lbl_col].str.upper()
# Keep only rows with equal lengths
m = df[seq_col].str.len() == df[lbl_col].str.len()
df = df[m]
# Filter very long sequences for GPU efficiency
df = df[df[seq_col].str.len() <= CFG['data']['max_seq_len_filter']].reset_index(drop=True)
print('After clean/filter:', df.shape, '| mean len:', df[seq_col].str.len().mean())
# Optional: length-stratified split
from sklearn.model_selection import train_test_split
if CFG['data'].get('length_stratified_split', False):
    lens = df[seq_col].str.len()
    bins = pd.qcut(lens, q=10, duplicates='drop')
    df_train, df_tmp = train_test_split(df, test_size=1-CFG['data']['train_fraction'], stratify=bins, random_state=CFG['seed'])
    lens_tmp = df_tmp[seq_col].str.len(); bins_tmp = pd.qcut(lens_tmp, q=10, duplicates='drop')
    val_size = CFG['data']['val_fraction'] / (1 - CFG['data']['train_fraction'])
    df_val, df_test = train_test_split(df_tmp, test_size=1-val_size, stratify=bins_tmp, random_state=CFG['seed'])
else:
    df = df.sample(frac=1.0, random_state=CFG['seed']).reset_index(drop=True)
    n = len(df)
    n_train = int(n * CFG['data']['train_fraction'])
    n_val = int(n * CFG['data']['val_fraction'])
    df_train = df.iloc[:n_train]
    df_val = df.iloc[n_train:n_train+n_val]
    df_test = df.iloc[n_train+n_val:]
print('Splits:', len(df_train), len(df_val), len(df_test))
target = CFG['data']['target']
num_classes = 8 if target=='q8' else 3
vocab = build_vocab(CFG['model']['vocab'])
print('Vocab size (including PAD):', len(vocab)+1, '| Classes:', num_classes)

Merged: (20990, 4)
After clean/filter: (16758, 4) | mean len: 230.4144289294665
Splits: 13406 1675 1677
Vocab size (including PAD): 22 | Classes: 8


## Dataset and DataLoaders

In [22]:
class ProteinSSDataset(Dataset):
    def __init__(self, df, vocab, seq_column, label_column, target='q8'):
        self.seqs = []
        self.labels = []
        for _,row in df.iterrows():
            s = encode_sequence(row[seq_column], vocab)
            l = encode_labels(row[label_column], target)
            n = min(len(s), len(l))
            self.seqs.append(s[:n]); self.labels.append(l[:n])
    def __len__(self): return len(self.seqs)
    def __getitem__(self, idx): return self.seqs[idx], self.labels[idx]

class PadCollator:
    def __init__(self, max_len): self.max_len = max_len
    def __call__(self, batch):
        seqs,labels = zip(*batch)
        x,y,mask,lengths = pad_batch(seqs, labels, self.max_len)
        return {'input_ids': x, 'labels': y, 'mask': mask, 'lengths': lengths}

ds_train = ProteinSSDataset(df_train, vocab, CFG['data']['seq_column'], lbl_col, target)
ds_val   = ProteinSSDataset(df_val, vocab, CFG['data']['seq_column'], lbl_col, target)
ds_test  = ProteinSSDataset(df_test, vocab, CFG['data']['seq_column'], lbl_col, target)
collate = PadCollator(CFG['data']['max_len'])
loader_train = DataLoader(ds_train, batch_size=CFG['training']['batch_size'], shuffle=True, num_workers=CFG['training']['num_workers'], pin_memory=CFG['training']['pin_memory'], collate_fn=collate)
loader_val   = DataLoader(ds_val,   batch_size=CFG['training']['batch_size'], shuffle=False, num_workers=CFG['training']['num_workers'], pin_memory=CFG['training']['pin_memory'], collate_fn=collate)
loader_test  = DataLoader(ds_test,  batch_size=CFG['training']['batch_size'], shuffle=False, num_workers=CFG['training']['num_workers'], pin_memory=CFG['training']['pin_memory'], collate_fn=collate)
len(ds_train), len(ds_val), len(ds_test)

(13406, 1675, 1677)

## Model: BiLSTM with optional Self-Attention refinement

In [23]:
class BiLSTM_PSSP(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx=0, dropout=0.3, layers=2, attn=False, attn_heads=4, attn_dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.bilstm1 = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(hidden_dim*2, hidden_dim, bidirectional=True, batch_first=True)
        self.use_attn = attn
        if attn:
            self.attn = nn.MultiheadAttention(embed_dim=hidden_dim*2, num_heads=attn_heads, dropout=attn_dropout, batch_first=True)
            self.ln = nn.LayerNorm(hidden_dim*2)
        self.classifier = nn.Linear(hidden_dim*2, num_classes)
    def forward(self, x, mask=None):
        x = self.embedding(x)
        x,_ = self.bilstm1(x)
        x = self.dropout(x)
        x,_ = self.bilstm2(x)
        if self.use_attn:
            # key_padding_mask: True for PAD positions
            key_pad = None
            if mask is not None: key_pad = ~mask
            attn_out,_ = self.attn(x, x, x, key_padding_mask=key_pad, need_weights=False)
            x = self.ln(x + attn_out)
        out = self.classifier(x)
        return out

vocab_size = len(vocab) + 1  # + PAD
model = BiLSTM_PSSP(
    vocab_size=vocab_size,
    embed_dim=CFG['model']['embedding_dim'],
    hidden_dim=CFG['model']['hidden_dim'],
    num_classes=num_classes,
    pad_idx=PAD_IDX,
    dropout=CFG['model']['dropout'],
    layers=CFG['model']['layers'],
    attn=CFG['model']['attention'],
    attn_heads=CFG['model']['attn_heads'],
    attn_dropout=CFG['model']['attn_dropout'],
).to(device)
model

BiLSTM_PSSP(
  (embedding): Embedding(22, 128, padding_idx=0)
  (bilstm1): LSTM(128, 256, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (bilstm2): LSTM(512, 256, batch_first=True, bidirectional=True)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
  )
  (ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=512, out_features=8, bias=True)
)

## Training and validation loops

In [24]:
params = [p for p in model.parameters() if p.requires_grad]
if CFG['training']['optimizer'].lower()=='adamw':
    optimizer = torch.optim.AdamW(params, lr=CFG['training']['lr'], weight_decay=CFG['training']['weight_decay'])
else:
    optimizer = torch.optim.Adam(params, lr=CFG['training']['lr'])
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX, label_smoothing=CFG['model']['label_smoothing'])
scheduler = None
if CFG['training']['reduce_lr_on_plateau']['enabled']:
    r = CFG['training']['reduce_lr_on_plateau']
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=r['factor'], patience=r['patience'])
scaler = GradScaler(enabled=CFG['training']['mixed_precision'])

def train_one_epoch(model, loader):
    model.train(); total_loss=0.0; total_acc=0.0; steps=0
    for batch in tqdm(loader, desc='Train', leave=False):
        x = batch['input_ids'].to(device)
        y = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        optimizer.zero_grad(set_to_none=True)
        if CFG['training']['mixed_precision']:
            with autocast():
                logits = model(x, mask=mask)
                loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            scaler.scale(loss).backward()
            if CFG['training']['grad_clip_norm']:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG['training']['grad_clip_norm'])
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(x, mask=mask)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss.backward()
            if CFG['training']['grad_clip_norm']:
                nn.utils.clip_grad_norm_(model.parameters(), CFG['training']['grad_clip_norm'])
            optimizer.step()
        acc = masked_accuracy(logits, y, mask)
        total_loss += loss.item(); total_acc += acc; steps += 1
    return total_loss/max(1,steps), total_acc/max(1,steps)

@torch.no_grad()
def validate(model, loader):
    model.eval(); total_loss=0.0; total_acc=0.0; steps=0
    all_logits=[]; all_labels=[]; all_masks=[]
    for batch in tqdm(loader, desc='Val', leave=False):
        x = batch['input_ids'].to(device)
        y = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        logits = model(x, mask=mask)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        acc = masked_accuracy(logits, y, mask)
        total_loss += loss.item(); total_acc += acc; steps += 1
        all_logits.append(logits.cpu()); all_labels.append(y.cpu()); all_masks.append(mask.cpu())
    return total_loss/max(1,steps), total_acc/max(1,steps), (torch.cat(all_logits,0), torch.cat(all_labels,0), torch.cat(all_masks,0))

best_val_loss = float('inf'); epochs_no_improve = 0
best_state = None
for epoch in range(1, CFG['training']['epochs']+1):
    print('Epoch {}/{}'.format(epoch, CFG['training']['epochs']))
    tr_loss, tr_acc = train_one_epoch(model, loader_train)
    val_loss, val_acc, _ = validate(model, loader_val)
    if scheduler is not None: scheduler.step(val_loss)
    print(f'  train_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={val_loss:.4f} acc={val_acc:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss; epochs_no_improve = 0
        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        torch.save(model.state_dict(), CFG['output']['best_model_path'].replace('${target}', target))
        print('  Saved best checkpoint.')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= CFG['training']['early_stopping_patience']:
            print('  Early stopping.')
            break
# Load best
if best_state is not None: model.load_state_dict(best_state)
gc.collect(); torch.cuda.empty_cache()

/var/tmp/pbs.12242590.pbs101/ipykernel_2487179/1800919510.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=CFG['training']['mixed_precision'])


Epoch 1/40


Train:   0%|          | 0/419 [00:00<?, ?it/s]/var/tmp/pbs.12242590.pbs101/ipykernel_2487179/1800919510.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
                                                        

RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 700 but got size 481 for tensor number 1 in the list.

## Final evaluation on test set

In [ ]:
test_loss, test_acc, (logits, labels, masks) = validate(model, loader_test)
res = evaluate_logits(logits, labels, masks, num_classes)
print('Test Metrics:')
print('Q{} Accuracy: {:.2f} %'.format(num_classes, res['accuracy']*100))
print('Precision (mean): {:.4f}'.format(np.mean(res['precision'])))
print('Recall (mean): {:.4f}'.format(np.mean(res['recall'])))
print('F1 (mean): {:.4f}'.format(np.mean(res['f1'])))
print('SOV: {:.4f}'.format(res['sov']))
# Confusion matrix plot
plt.figure(figsize=(5,4))
plt.imshow(res['confusion_matrix'], cmap='Blues')
plt.colorbar(); plt.title('Confusion Matrix')
plt.xlabel('Pred'); plt.ylabel('True'); plt.tight_layout(); plt.show()

## Optional: quick per-sequence visualization

In [ ]:
# Visualize predictions vs ground-truth for a random test protein
idx = np.random.randint(len(ds_test))
seq_ids, gt = ds_test[idx]
x = torch.tensor(seq_ids, dtype=torch.long).unsqueeze(0).to(device)
m = torch.ones_like(x, dtype=torch.bool).to(device)
with torch.no_grad():
    out = model(x, mask=m).argmax(-1).squeeze(0).detach().cpu().numpy()
plt.figure(figsize=(12,2))
plt.plot(gt, label='GT', alpha=0.8); plt.plot(out[:len(gt)], label='Pred', alpha=0.6)
plt.yticks(range(num_classes)); plt.legend(); plt.title('Per-residue labels (indices)'); plt.show()